# 01 — Tải QM9
Tải QM9 (133 885 phân tử hữu cơ nhỏ C/H/N/O/F) và `uncharacterized.txt` (3 054 phân tử không qua kiểm tra nhất quán) từ Figshare, rồi đọc một lượt để giữ các phân tử có ≤ `MAX_HEAVY` nguyên tử nặng.

Hình học QM9 là B3LYP/6-31G(2df,p); ở đây chỉ dùng làm hình học đầu vào, mọi năng lượng sẽ được tính lại bằng PySCF.

**Đầu ra:** `data/raw/dsgdb9nsd.xyz.tar.bz2`, `uncharacterized.txt`, `qm9_candidates.jsonl`, `qm9_summary.json`.

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import pandas as pd
from utils import data_utils as du

MAX_HEAVY = 5
TAR = f"{RAW}/dsgdb9nsd.xyz.tar.bz2"
UNCHAR = f"{RAW}/uncharacterized.txt"

## 1. Tải file (bỏ qua nếu đã có)

In [ ]:
t0 = time.time()
du.download(du.QM9_URL, TAR)
du.download(du.UNCHAR_URL, UNCHAR)
print(f"tar: {os.path.getsize(TAR)/1e6:.1f} MB | uncharacterized: {os.path.getsize(UNCHAR)/1e3:.1f} kB | {time.time()-t0:.0f}s")

## 2. Đọc tar một lượt, giữ phân tử ≤ MAX_HEAVY nguyên tử nặng

In [ ]:
t0 = time.time()
cands, stats = du.scan_qm9_tar(TAR, max_heavy=MAX_HEAVY, out_jsonl=f"{RAW}/qm9_candidates.jsonl")
bad = du.read_uncharacterized(UNCHAR)
stats["n_uncharacterized_listed"] = len(bad)
stats["n_candidates_uncharacterized"] = sum(c["gdb_id"] in bad for c in cands)
json.dump(stats, open(f"{RAW}/qm9_summary.json", "w"), indent=2)
print(f"{time.time()-t0:.0f}s | tổng {stats['n_total']} phân tử | ứng viên (≤{MAX_HEAVY} nguyên tử nặng): {stats['n_candidates']}")
print("Số phân tử theo số nguyên tử nặng:", stats["by_n_heavy"])
print(f"uncharacterized.txt liệt kê {len(bad)} phân tử (kỳ vọng ≈3054); trong ứng viên: {stats['n_candidates_uncharacterized']}")
if not (2500 < len(bad) < 3500):
    print("CẢNH BÁO: số phân tử uncharacterized bất thường, kiểm tra lại cách đọc file.")

In [ ]:
df = pd.DataFrame(cands)[["gdb_id", "formula", "n_heavy", "n_atoms", "smiles", "U0_B3LYP_Ha"]]
df.head(10)